# Build an Agent

By themselves, language models can't take actions - they just output text.
A big use case for LangChain is creating **agents**.
[Agents](/docs/concepts/agents) are systems that use [LLMs](/docs/concepts/chat_models) as reasoning engines to determine which actions to take and the inputs necessary to perform the action.
After executing actions, the results can be fed back into the LLM to determine whether more actions are needed, or whether it is okay to finish. This is often achieved via [tool-calling](/docs/concepts/tool_calling).

In this tutorial we will build an agent that can interact with a search engine. You will be able to ask this agent questions, watch it call the search tool, and have conversations with it.

## End-to-end agent

The code snippet below represents a fully functional agent that uses an LLM to decide which tools to use. It is equipped with a generic search tool. It has conversational memory - meaning that it can be used as a multi-turn chatbot.

In the rest of the guide, we will walk through the individual components and what each part does - but if you want to just grab some code and get started, feel free to use this!

In [ ]:
# Import relevant functionality
#from langchain_anthropic import ChatAnthropic
from langchain_openai import ChatOpenAI

from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import MemorySaver
from langchain.agents import create_agent

# Create the agent
memory = MemorySaver()

#model = ChatAnthropic(model_name="claude-3-sonnet-20240229")
model = ChatOpenAI(model="gpt-4o-mini")


search = TavilySearchResults(max_results=2)
tools = [search]
agent_executor = create_agent(model, tools, checkpointer=memory)

# Use the agent
config = {"configurable": {"thread_id": "abc123"}}
for chunk in agent_executor.stream(
    {"messages": [HumanMessage(content="hi im bob! and i live in sf")]}, config
):
    print(chunk)
    print("----")

for chunk in agent_executor.stream(
    {"messages": [HumanMessage(content="whats the weather where I live?")]}, config
):
    print(chunk)
    print("----")

## Setup

### Jupyter Notebook

This guide (and most of the other guides in the documentation) uses [Jupyter notebooks](https://jupyter.org/) and assumes the reader is as well. Jupyter notebooks are perfect interactive environments for learning how to work with LLM systems because oftentimes things can go wrong (unexpected output, API down, etc), and observing these cases is a great way to better understand building with LLMs.

This and other tutorials are perhaps most conveniently run in a Jupyter notebook. See [here](https://jupyter.org/install) for instructions on how to install.

### Installation

To install LangChain run:

In [ ]:
%pip install -U langchain-community langgraph langchain-openai tavily-python langgraph-checkpoint-sqlite

For more details, see our [Installation guide](/docs/how_to/installation).

### LangSmith

Many of the applications you build with LangChain will contain multiple steps with multiple invocations of LLM calls.
As these applications get more and more complex, it becomes crucial to be able to inspect what exactly is going on inside your chain or agent.
The best way to do this is with [LangSmith](https://smith.langchain.com).

After you sign up at the link above, make sure to set your environment variables to start logging traces:

```shell
export LANGCHAIN_TRACING_V2="true"
export LANGCHAIN_API_KEY="..."
```

Or, if in a notebook, you can set them with:

```python
import getpass
import os

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass()
```

### Tavily

We will be using [Tavily](/docs/integrations/tools/tavily_search) (a search engine) as a tool.
In order to use it, you will need to get and set an API key:

```bash
export TAVILY_API_KEY="..."
```

Or, if in a notebook, you can set it with:

```python
import getpass
import os

os.environ["TAVILY_API_KEY"] = getpass.getpass()
```

## Define tools

We first need to create the tools we want to use. Our main tool of choice will be [Tavily](/docs/integrations/tools/tavily_search) - a search engine. We have a built-in tool in LangChain to easily use Tavily search engine as tool.


In [ ]:
from langchain_community.tools.tavily_search import TavilySearchResults

search = TavilySearchResults(max_results=2)
search_results = search.invoke("what is the weather in SF")
print(search_results)
# If we want, we can create other tools.
# Once we have all the tools we want, we can put them in a list that we will reference later.
tools = [search]

## Using Language Models

Next, let's learn how to use a language model by to call tools. LangChain supports many different language models that you can use interchangably - select the one you want to use below!

import ChatModelTabs from "@theme/ChatModelTabs";

<ChatModelTabs openaiParams={`model="gpt-4"`} />


In [ ]:
# | output: false
# | echo: false

#from langchain_anthropic import ChatAnthropic
#model = ChatAnthropic(model="claude-3-sonnet-20240229")

from langchain_openai import ChatOpenAI
model = ChatOpenAI(model="gpt-4o-mini")


You can call the language model by passing in a list of messages. By default, the response is a `content` string.

In [ ]:
from langchain_core.messages import HumanMessage

response = model.invoke([HumanMessage(content="hi!")])
response.content

We can now see what it is like to enable this model to do tool calling. In order to enable that we use `.bind_tools` to give the language model knowledge of these tools

In [ ]:
model_with_tools = model.bind_tools(tools)

We can now call the model. Let's first call it with a normal message, and see how it responds. We can look at both the `content` field as well as the `tool_calls` field.

In [ ]:
response = model_with_tools.invoke([HumanMessage(content="Hi!")])

print(f"ContentString: {response.content}")
print(f"ToolCalls: {response.tool_calls}")

Now, let's try calling it with some input that would expect a tool to be called.

In [ ]:
response = model_with_tools.invoke([HumanMessage(content="What's the weather in SF?")])

print(f"ContentString: {response.content}")
print(f"ToolCalls: {response.tool_calls}")

We can see that there's now no text content, but there is a tool call! It wants us to call the Tavily Search tool.

This isn't calling that tool yet - it's just telling us to. In order to actually call it, we'll want to create our agent.

## Create the agent

Now that we have defined the tools and the LLM, we can create the agent. We will be using [LangGraph](/docs/concepts/architecture/#langgraph) to construct the agent. 
Currently, we are using a high level interface to construct the agent, but the nice thing about LangGraph is that this high-level interface is backed by a low-level, highly controllable API in case you want to modify the agent logic.


Now, we can initialize the agent with the LLM and the tools.

Note that we are passing in the `model`, not `model_with_tools`. That is because `create_react_agent` will call `.bind_tools` for us under the hood.

In [ ]:
from langchain.agents import create_agent

agent_executor = create_agent(model, tools)

## Run the agent

We can now run the agent with a few queries! Note that for now, these are all **stateless** queries (it won't remember previous interactions). Note that the agent will return the **final** state at the end of the interaction (which includes any inputs, we will see later on how to get only the outputs).

First up, let's see how it responds when there's no need to call a tool:

In [ ]:
response = agent_executor.invoke({"messages": [HumanMessage(content="hi!")]})

response["messages"]

In order to see exactly what is happening under the hood (and to make sure it's not calling a tool) we can take a look at the [LangSmith trace](https://smith.langchain.com/public/28311faa-e135-4d6a-ab6b-caecf6482aaa/r)

Let's now try it out on an example where it should be invoking the tool

In [ ]:
response = agent_executor.invoke(
    {"messages": [HumanMessage(content="whats the weather in sf?")]}
)
response["messages"]

We can check out the [LangSmith trace](https://smith.langchain.com/public/f520839d-cd4d-4495-8764-e32b548e235d/r) to make sure it's calling the search tool effectively.

## Streaming Messages

We've seen how the agent can be called with `.invoke` to get  a final response. If the agent executes multiple steps, this may take a while. To show intermediate progress, we can stream back messages as they occur.

In [ ]:
for chunk in agent_executor.stream(
    {"messages": [HumanMessage(content="whats the weather in sf?")]}
):
    print(chunk)
    print("----")

## Streaming tokens

In addition to streaming back messages, it is also useful to stream back tokens.
We can do this with the `.astream_events` method.

:::important
This `.astream_events` method only works with Python 3.11 or higher.
:::

In [ ]:
async for event in agent_executor.astream_events(
    {"messages": [HumanMessage(content="whats the weather in sf?")]}, version="v2"
):
    kind = event["event"]
    if kind == "on_chain_start":
        if (
            event["name"] == "Agent"
        ):  # Was assigned when creating the agent with `.with_config({"run_name": "Agent"})`
            print(
                f"Starting agent: {event['name']} with input: {event['data'].get('input')}"
            )
    elif kind == "on_chain_end":
        if (
            event["name"] == "Agent"
        ):  # Was assigned when creating the agent with `.with_config({"run_name": "Agent"})`
            print()
            print("--")
            print(
                f"Done agent: {event['name']} with output: {event['data'].get('output')['output']}"
            )
    if kind == "on_chat_model_stream":
        content = event["data"]["chunk"].content
        if content:
            # Empty content in the context of OpenAI means
            # that the model is asking for a tool to be invoked.
            # So we only print non-empty content
            print(content, end="|")
    elif kind == "on_tool_start":
        print("--")
        print(
            f"Starting tool: {event['name']} with inputs: {event['data'].get('input')}"
        )
    elif kind == "on_tool_end":
        print(f"Done tool: {event['name']}")
        print(f"Tool output was: {event['data'].get('output')}")
        print("--")

## Adding in memory

As mentioned earlier, this agent is stateless. This means it does not remember previous interactions. To give it memory we need to pass in a checkpointer. When passing in a checkpointer, we also have to pass in a `thread_id` when invoking the agent (so it knows which thread/conversation to resume from).

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()

In [ ]:
agent_executor = create_agent(model, tools, checkpointer=memory)

config = {"configurable": {"thread_id": "abc123"}}

In [ ]:
for chunk in agent_executor.stream(
    {"messages": [HumanMessage(content="hi im bob!")]}, config
):
    print(chunk)
    print("----")

In [ ]:
for chunk in agent_executor.stream(
    {"messages": [HumanMessage(content="whats my name?")]}, config
):
    print(chunk)
    print("----")

Example [LangSmith trace](https://smith.langchain.com/public/fa73960b-0f7d-4910-b73d-757a12f33b2b/r)

If you want to start a new conversation, all you have to do is change the `thread_id` used

In [ ]:
config = {"configurable": {"thread_id": "xyz123"}}
for chunk in agent_executor.stream(
    {"messages": [HumanMessage(content="whats my name?")]}, config
):
    print(chunk)
    print("----")

## Conclusion

That's a wrap! In this quick start we covered how to create a simple agent. 
We've then shown how to stream back a response - not only with the intermediate steps, but also tokens!
We've also added in memory so you can have a conversation with them.
Agents are a complex topic with lots to learn! 

For more information on Agents, please check out the [LangGraph](/docs/concepts/architecture/#langgraph) documentation. This has it's own set of concepts, tutorials, and how-to guides.

### SD Response

In [ ]:
# Use the personalized agent
config2 = {"configurable": {"thread_id": "abc456"}}
for chunk in agent_executor.stream(
    {"messages": [HumanMessage(content="hi im shishir! and i live in seattle")]}, config2
):
    print(chunk)
    print("----")

for chunk in agent_executor.stream(
    {"messages": [HumanMessage(content="whats the weather where I live?")]}, config2
):
    print(chunk)
    print("----")

### SD Module 3: Class Discussion

In [ ]:
disc3prompt = """Compare LangChain and LangGraph for designing LLM-based agents with memory, adaptability, and complex workflow execution in 'Cloud Computing and eCommerce'.

Discuss integration considerations, handling challenges like hallucinations, and ensuring reliability in the agent design for 'Cloud Computing and eCommerce.'"""

In [ ]:
# | output: false
# | echo: false

from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4o-mini")

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage
response = model.invoke([HumanMessage(disc3prompt)])
print(response.content)

When comparing LangChain and LangGraph for designing LLM-based agents with memory, adaptability, and complex workflow execution in the domains of Cloud Computing and eCommerce, both frameworks offer unique strengths and features. However, they also bring specific considerations regarding integration, challenges like hallucinations, and ensuring reliability.

### Overview of LangChain and LangGraph

**LangChain:**
- LangChain is a versatile framework tailored for developing applications using Large Language Models (LLMs). It offers tools for building agents with memory, prompts, chains, and tool integrations.
- Key features include modularity, allowing developers to compose workflows easily, and a rich ecosystem of pre-built modules.
- It offers robust mechanisms for memory management, making it easier to create persistent agents that learn from interactions over time.

**LangGraph:**
- LangGraph emphasizes graph-based structuring for workflows, mainly focusing on state management and complex interactions in LLM applications.
- It promotes adaptability through a visual-oriented approach to building agent workflows, which can be particularly useful for managing interactions in dynamic environments like eCommerce.
- Memory capabilities come from leveraging the graph structure, which allows for multi-faceted relationships between nodes (representing states or interactions).

### Integration Considerations

1. **Ecosystem Compatibility:**
   - **LangChain** often integrates seamlessly with tools like retrievers and databases. This could be advantageous for a cloud computing or eCommerce agent needing to access large datasets.
   - **LangGraph** could be more effective in systems where data relationships are complex, such as supply chains in eCommerce or multi-layered cloud service offerings.

2. **APIs and Microservices:**
   - Both frameworks facilitate API integrations, but LangChain might lead to higher modularity, allowing for easier updates and adjustments to specific components of the agent with minimal disruption.
   - For LangGraph, ensure that the visual workflows can accommodate third-party APIs used in eCommerce or cloud services effectively.

3. **Scalability:**
   - In terms of scalability, LangChain’s modularity may offer better pathways for scaling up specific functionalities, while LangGraph's graph-based approach might be more suitable for expanding the complexity of relationships in services.

### Handling Challenges (e.g., Hallucinations)

- **LangChain:**
  - Implementing forms of structured prompt engineering can reduce hallucinations. LangChain allows for embedding retrieval-augmented generation strategies to provide contextually relevant information, thus reducing the risk of inaccurate outputs.
  - Building in-memory capabilities can help refine and contextualize responses over time, further preventing hallucinations.

- **LangGraph:**
  - The graph-based architecture may inherently manage context better, as nodes can store historical interactions, which inform future outputs. This could lead to reducing hallucinations via contextual awareness.
  - However, it may require additional effort to ensure that the flow of information doesn’t lead to incorrect relationships being formed in complex workflows.

### Ensuring Reliability in Agent Design

1. **Validation Mechanisms:**
   - Implement validation steps within both frameworks to ensure that outputs meet the expected standards before interacting with users or systems.
   - LangChain's chains can include validation layers that check for sensible outputs, while LangGraph can leverage its architectural features to monitor states dynamically.

2. **Error Handling and Retries:**
   - Robust error handling must be inherent in both frameworks. For LangChain, using built-in retry mechanisms can help in recovering from failures quickly.
   - LangGraph's reactive state management can also help in recovering from errors through state transitions.

3. **Testing and Monitoring:**
   - Continuous testing should be a practice within both ecosystems, with a focus on monitoring performance in real-world applications in eCommerce and cloud computing contexts.
   - Implementing logging and analytics will help track decision-making patterns and identify points of failure due to hallucinations or incorrect data flows.

### Conclusion

Both LangChain and LangGraph offer unique attributes beneficial for developing LLM-based agents in cloud computing and eCommerce. LangChain shines in modularity and extensive tool integration, suitable for flexible and scalable applications. LangGraph excels in representing complex interdependencies visually, making it powerful for applications with intricate workflows. Ultimately, the choice between the two will depend on the specific requirements of the application, including the need for adaptability, memory management, and workflow complexity. Careful consideration of integration, error handling, and validation mechanisms can lead to the successful deployment of reliable agents within these domains.

In [ ]:
disc3prompt_peer = """Compare LangChain and LangGraph for designing LLM-based agents with memory, adaptability, and complex workflow execution in Health Care.

Discuss integration considerations, handling challenges like hallucinations, and ensuring reliability in the agent design for Health Care."""

In [ ]:
response = model.invoke([HumanMessage(disc3prompt_peer)])
print(response.content)

LangChain and LangGraph are both frameworks that facilitate the development of language model (LLM)-based agents, but they come with distinct features, strengths, and weaknesses that can impact their application in healthcare settings, particularly when designing agents with memory, adaptability, and complex workflow execution. Below is a comprehensive comparison and discussion on their merits, integration considerations, and strategies for handling challenges:

### LangChain

**Overview**: 
LangChain is designed to empower the creation of applications that can utilize LLMs to perform various tasks such as summarization, question answering, and text generation. It has built-in memory management capabilities and tools for chaining together different tasks and components.

**Strengths**:
1. **Memory Management**: LangChain offers memory modules that allow agents to retain relevant information across sessions, making it suitable for personalized patient interactions and ongoing scheduling or treatment plans.
2. **Modularity**: The framework allows for modular design, wherein logic can be encapsulated in chains, making it easier to handle complex workflows.
3. **Integration with APIs**: LangChain supports integration with various APIs and data sources, which is crucial for healthcare applications that often require real-time data access.

**Challenges**:
1. **Handling Hallucinations**: Users need to implement additional layers of validation to mitigate the risk of the model generating information that seems plausible but is incorrect. This often involves defining strict data pipelines and feedback loops.
2. **Reliability**: Given the critical nature of healthcare, integrating fail-safes and regular checks within the design is necessary to ensure that responses meet clinical accuracy standards.

**Integration Considerations**:
- Interoperability with electronic health records (EHRs): Ensure that LangChain integrates seamlessly with common EHR systems to retrieve and update patient data.
- Compliance with health regulations: Adhere to HIPAA and other privacy regulations, ensuring that any data processed by LangChain remains secure.

### LangGraph

**Overview**: 
LangGraph is more focused on the representation of knowledge as graphs, promoting the idea of relational data structures that can be more intuitive than linear processing pipelines.

**Strengths**:
1. **Graph-Based Memory**: The ability to represent information as a graph allows for more complex relationships and context around patient data, which can enhance adaptability during interactions.
2. **Contextual Awareness**: LangGraph's design enables dynamic retrieval of information relevant to patient interactions based on the graph's structure, fostering more nuanced conversations and responses.
3. **Complex Workflow Execution**: The framework is well-suited for representing workflows graphically, making it easier to visualize and execute complex healthcare protocols across various agents.

**Challenges**:
1. **Integration Complexity**: While the graph structure is powerful, it can introduce complexity in the integration with existing healthcare IT systems, which are often relational databases.
2. **Scalability**: As the amount of data grows, ensuring that LangGraph efficiently manages large and complex graphs poses a scaling challenge.

**Integration Considerations**:
- Ensuring that the graph database used is compatible with existing infrastructure and meets performance benchmarks required for healthcare applications.
- Addressing regulatory concerns, similar to LangChain, particularly around data access and patient privacy.

### Addressing Challenges Like Hallucinations & Reliability

1. **Validation Layers**: Both frameworks should incorporate robust validation mechanisms, such as rule-based checks or human-in-the-loop systems, to review outputs from the LLM prior to presenting them to healthcare providers or patients.

2. **Feedback Mechanisms**: Continuous learning can be implemented through feedback loops that update the agent based on user interactions, correcting errors and improving decision-making over time.

3. **Audit Trails**: Documentation of interactions and decisions made by the agent can provide clarity and accountability, which is essential in healthcare environments.

4. **Decision Support Systems**: Integrating decision support tools can supplement the model’s guidance, helping to ensure that suggestions and responses are clinically appropriate.

### Conclusion

Both LangChain and LangGraph offer valuable frameworks for developing LLM-based agents in healthcare, but their suitability will depend on the specific needs of the application. LangChain's modularity and memory features lend themselves well to patient interaction systems, while LangGraph's ability to handle complex workflows and relationships may better suit applications that require a high degree of contextual understanding. 

Regardless of the chosen framework, addressing the challenges of hallucinations and ensuring reliability through validation, auditing, and compliance with healthcare regulations is paramount to successfully deploying LLM-based agents in the healthcare domain.